# Week 3 — Integrated Capstone Project
## Forward/Backward Selection, PCR, and PLSR

**Datasets:** Cervical Cancer Risk Factors and Breast Cancer Wisconsin Diagnostic

### Central question
How do forward and backward feature selection, Principal Component Regression (PCR), and Partial Least Squares Regression (PLSR) compare with ordinary linear regression when predicting a continuous outcome from potentially correlated predictors?

This notebook applies the same workflow to both project datasets.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def metrics(model, X, y):
    pred = model.predict(X).ravel()
    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": mean_squared_error(y, pred) ** 0.5,
        "R2": r2_score(y, pred)
    }

def cv_rmse(model, X, y):
    s = -cross_val_score(model, X, y, cv=CV,
                         scoring="neg_root_mean_squared_error")
    return s.mean(), s.std()


## 1. Cervical Cancer Risk Factors

The Week 2 continuous target is retained so Week 3 can be compared with the earlier analysis. The notebook uses numeric predictors and handles missing values using training-set medians.


In [ ]:
cervical = pd.read_csv("risk_factors_cervical_cancer.csv")
cervical = cervical.replace(["?", ""], np.nan)

for col in cervical.columns:
    cervical[col] = pd.to_numeric(cervical[col], errors="coerce")

target = "Age"
Xc = cervical.drop(columns=[target])
yc = cervical[target]

Xc = Xc.select_dtypes(include=np.number)
valid = yc.notna()
Xc, yc = Xc.loc[valid], yc.loc[valid]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=.20, random_state=RANDOM_STATE
)

Xc_train = Xc_train.fillna(Xc_train.median())
Xc_test = Xc_test.fillna(Xc_train.median())

print(Xc_train.shape, Xc_test.shape)


### 1.1 Ordinary Linear Regression Baseline


In [ ]:
baseline_c = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression())
])
baseline_c.fit(Xc_train, yc_train)
baseline_c_metrics = metrics(baseline_c, Xc_test, yc_test)
baseline_c_metrics


### 1.2 Forward Selection


In [ ]:
forward_c = SequentialFeatureSelector(
    LinearRegression(),
    direction="forward",
    n_features_to_select="auto",
    tol=0.01,
    scoring="neg_root_mean_squared_error",
    cv=CV,
    n_jobs=-1
)
forward_c.fit(Xc_train, yc_train)

forward_features_c = Xc_train.columns[forward_c.get_support()].tolist()
print("Selected features:", forward_features_c)

forward_model_c = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression())
])
forward_model_c.fit(Xc_train[forward_features_c], yc_train)
forward_metrics_c = metrics(
    forward_model_c, Xc_test[forward_features_c], yc_test
)
forward_metrics_c


### 1.3 Backward Selection


In [ ]:
backward_c = SequentialFeatureSelector(
    LinearRegression(),
    direction="backward",
    n_features_to_select="auto",
    tol=0.01,
    scoring="neg_root_mean_squared_error",
    cv=CV,
    n_jobs=-1
)
backward_c.fit(Xc_train, yc_train)

backward_features_c = Xc_train.columns[backward_c.get_support()].tolist()
print("Selected features:", backward_features_c)

backward_model_c = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression())
])
backward_model_c.fit(Xc_train[backward_features_c], yc_train)
backward_metrics_c = metrics(
    backward_model_c, Xc_test[backward_features_c], yc_test
)
backward_metrics_c


### 1.4 Principal Component Regression (PCR)


In [ ]:
pcr_results_c = []
for n in range(1, min(Xc_train.shape[1], 20) + 1):
    model = Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=n)),
        ("regression", LinearRegression())
    ])
    mean_rmse, sd_rmse = cv_rmse(model, Xc_train, yc_train)
    pcr_results_c.append([n, mean_rmse, sd_rmse])

pcr_table_c = pd.DataFrame(
    pcr_results_c, columns=["Components", "CV_RMSE", "CV_SD"]
)
pcr_table_c


In [ ]:
best_pcr_c = int(
    pcr_table_c.loc[pcr_table_c["CV_RMSE"].idxmin(), "Components"]
)

pcr_c = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=best_pcr_c)),
    ("regression", LinearRegression())
])
pcr_c.fit(Xc_train, yc_train)
pcr_metrics_c = metrics(pcr_c, Xc_test, yc_test)

print("Best PCR components:", best_pcr_c)
pcr_metrics_c


### 1.5 Partial Least Squares Regression (PLSR)


In [ ]:
plsr_results_c = []
for n in range(1, min(Xc_train.shape[1], 20) + 1):
    model = Pipeline([
        ("scale", StandardScaler()),
        ("plsr", PLSRegression(n_components=n, scale=False))
    ])
    mean_rmse, sd_rmse = cv_rmse(model, Xc_train, yc_train)
    plsr_results_c.append([n, mean_rmse, sd_rmse])

plsr_table_c = pd.DataFrame(
    plsr_results_c, columns=["Components", "CV_RMSE", "CV_SD"]
)
plsr_table_c


In [ ]:
best_pls_c = int(
    plsr_table_c.loc[plsr_table_c["CV_RMSE"].idxmin(), "Components"]
)

plsr_c = Pipeline([
    ("scale", StandardScaler()),
    ("plsr", PLSRegression(n_components=best_pls_c, scale=False))
])
plsr_c.fit(Xc_train, yc_train)
plsr_metrics_c = metrics(plsr_c, Xc_test, yc_test)

print("Best PLS components:", best_pls_c)
plsr_metrics_c


## 2. Breast Cancer Wisconsin Diagnostic

The diagnosis label is excluded because this Week 3 assignment focuses on continuous-response regression. The continuous target `radius_mean` is represented by `feature_1` in the raw UCI WDBC file.


In [ ]:
wdbc_cols = ["id", "diagnosis"] + [f"feature_{i}" for i in range(1,31)]
breast = pd.read_csv("wdbc.data", header=None, names=wdbc_cols)

target_b = "feature_1"
Xb = breast.drop(columns=[target_b, "id", "diagnosis"])
yb = breast[target_b]

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb, yb, test_size=.20, random_state=RANDOM_STATE
)
Xb_train = Xb_train.fillna(Xb_train.median())
Xb_test = Xb_test.fillna(Xb_train.median())


### 2.1 Baseline, Forward Selection, and Backward Selection


In [ ]:
baseline_b = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression())
])
baseline_b.fit(Xb_train, yb_train)
baseline_b_metrics = metrics(baseline_b, Xb_test, yb_test)

forward_b = SequentialFeatureSelector(
    LinearRegression(), direction="forward", n_features_to_select="auto",
    tol=0.01, scoring="neg_root_mean_squared_error", cv=CV, n_jobs=-1
)
forward_b.fit(Xb_train, yb_train)
forward_features_b = Xb_train.columns[forward_b.get_support()].tolist()

forward_model_b = Pipeline([
    ("scale", StandardScaler()), ("model", LinearRegression())
])
forward_model_b.fit(Xb_train[forward_features_b], yb_train)
forward_metrics_b = metrics(
    forward_model_b, Xb_test[forward_features_b], yb_test
)

backward_b = SequentialFeatureSelector(
    LinearRegression(), direction="backward", n_features_to_select="auto",
    tol=0.01, scoring="neg_root_mean_squared_error", cv=CV, n_jobs=-1
)
backward_b.fit(Xb_train, yb_train)
backward_features_b = Xb_train.columns[backward_b.get_support()].tolist()

backward_model_b = Pipeline([
    ("scale", StandardScaler()), ("model", LinearRegression())
])
backward_model_b.fit(Xb_train[backward_features_b], yb_train)
backward_metrics_b = metrics(
    backward_model_b, Xb_test[backward_features_b], yb_test
)

print("Forward features:", forward_features_b)
print("Backward features:", backward_features_b)
print("Baseline:", baseline_b_metrics)
print("Forward:", forward_metrics_b)
print("Backward:", backward_metrics_b)


### 2.2 PCR


In [ ]:
pcr_results_b = []
for n in range(1, min(Xb_train.shape[1], 20) + 1):
    model = Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=n)),
        ("regression", LinearRegression())
    ])
    mean_rmse, sd_rmse = cv_rmse(model, Xb_train, yb_train)
    pcr_results_b.append([n, mean_rmse, sd_rmse])

pcr_table_b = pd.DataFrame(
    pcr_results_b, columns=["Components", "CV_RMSE", "CV_SD"]
)
best_pcr_b = int(
    pcr_table_b.loc[pcr_table_b["CV_RMSE"].idxmin(), "Components"]
)

pcr_b = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=best_pcr_b)),
    ("regression", LinearRegression())
])
pcr_b.fit(Xb_train, yb_train)
pcr_metrics_b = metrics(pcr_b, Xb_test, yb_test)

pcr_table_b, pcr_metrics_b


### 2.3 PLSR


In [ ]:
plsr_results_b = []
for n in range(1, min(Xb_train.shape[1], 20) + 1):
    model = Pipeline([
        ("scale", StandardScaler()),
        ("plsr", PLSRegression(n_components=n, scale=False))
    ])
    mean_rmse, sd_rmse = cv_rmse(model, Xb_train, yb_train)
    plsr_results_b.append([n, mean_rmse, sd_rmse])

plsr_table_b = pd.DataFrame(
    plsr_results_b, columns=["Components", "CV_RMSE", "CV_SD"]
)
best_pls_b = int(
    plsr_table_b.loc[plsr_table_b["CV_RMSE"].idxmin(), "Components"]
)

plsr_b = Pipeline([
    ("scale", StandardScaler()),
    ("plsr", PLSRegression(n_components=best_pls_b, scale=False))
])
plsr_b.fit(Xb_train, yb_train)
plsr_metrics_b = metrics(plsr_b, Xb_test, yb_test)

plsr_table_b, plsr_metrics_b


## 3. Model Comparison

Use the following cell after the analyses above to create a single comparison table. Report MAE, RMSE, and R² for each method and dataset.


In [ ]:
comparison = pd.DataFrame([
    ["Cervical", "Linear Regression", baseline_c_metrics],
    ["Cervical", "Forward Selection", forward_metrics_c],
    ["Cervical", "Backward Selection", backward_metrics_c],
    ["Cervical", "PCR", pcr_metrics_c],
    ["Cervical", "PLSR", plsr_metrics_c],
    ["Breast", "Linear Regression", baseline_b_metrics],
    ["Breast", "Forward Selection", forward_metrics_b],
    ["Breast", "Backward Selection", backward_metrics_b],
    ["Breast", "PCR", pcr_metrics_b],
    ["Breast", "PLSR", plsr_metrics_b],
], columns=["Dataset", "Method", "Metrics"])

comparison["MAE"] = comparison["Metrics"].apply(lambda d: d["MAE"])
comparison["RMSE"] = comparison["Metrics"].apply(lambda d: d["RMSE"])
comparison["R2"] = comparison["Metrics"].apply(lambda d: d["R2"])
comparison.drop(columns="Metrics").sort_values(["Dataset", "RMSE"])


## 4. Interpretation for Milestone One

Discuss:
- How many variables were retained by forward and backward selection?
- Did the two selection procedures retain similar predictors?
- How many components were selected for PCR and PLSR?
- How did PCR and PLSR compare with the full linear model?
- Did reducing the predictor space improve or reduce test-set performance?
- What does the comparison suggest about correlated predictors in each dataset?
- Which findings are consistent across both datasets?

**Methodological note:** Selection and component tuning use cross-validation on the training data. The test set is reserved for final evaluation to reduce test-set leakage.


## 5. Conclusion
This Week 3 notebook extends the Week 2 regularization analysis by examining stepwise feature selection and latent-variable methods. The results can be incorporated into Milestone One as evidence about model complexity, correlated predictors, feature selection, and dimensionality reduction.
